In [28]:
"""Data mapping module for standardizing and transforming suicide study dataset.
Handles data from both 2023 and 2013-2022 periods."""

import sys
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import numpy as np

from config.config import DATA_DIR

# Set output directory
output_file_path = DATA_DIR / 'mapped'

#================================================================================
# MAPPING DICTIONARIES
#================================================================================

# Column name mappings
COLUMN_MAPPINGS = {
    'ID samobójcy': 'ID',
    'Data rejestracji': 'Date',
    'Przedział wiekowy': 'AgeGroup',
    'Płeć': 'Gender',
    'Stan cywilny': 'Marital',
    'Wykształcenie': 'Education',
    'Informacje o pracy i nauce': 'WorkInfo',
    'Źródło utrzymania': 'Income',
    'Czy samobójstwo zakończyło się zgonem': 'Fatal',
    'Miejsce zamachu': 'Place',
    'Sposób popełnienia': 'Method',
    'Stan świadomości *': 'Substance',
    'Informacje dotyczące leczenia z powodu alkoholizmu/narkomanii': 'AbuseInfo1',
    'Powód zamachu *': 'Context1',
    'Powód zamachu 2': 'Context2',
    'Powód zamachu 3': 'Context3',
    'Powód zamachu 4': 'Context4',
    'Informacje dotyczące stanu zdrowia *': 'AbuseInfo2',
}

# Value mappings for each column
VALUE_MAPPINGS = {
    'AgeGroup': {
        "7-12": '07_12',
        '13-18': '13_18',
        '19-24': '19_24',
        '25-29': '25_29',
        '30-34': '30_34',
        '35-39': '35_39',
        '40-44': '40_44',
        '45-49': '45_49',
        '50-54': '50_54',
        '55-59': '55_59',
        '60-64': '60_64',
        '65-69': '65_69',
        '70-74': '70_74',
        '75-79': '75_79',
        '80-84': '80_84',
        '85+': '85',
        'Nieustalony wiek': np.nan
    },
    'Gender': {
        'M': 'M',
        "Mężczyzna": 'M',
        'K': 'F',
        "Kobieta": 'F',
    },
    'Marital': {
        'Kawaler/panna': 'Single',
        'Żonaty/zamężna': 'Married',
        'Wdowiec/wdowa': 'Widowed',
        'Rozwiedziony/rozwiedziona': 'Divorced',
        'W separacji': 'Separated',
        'Konkubent/konkubina': 'Cohabiting',
        'Brak danych/nieustalony': np.nan
    },
    'Education': {
        'Podstawowe': 'Primary',
        'Gimnazjalne': 'LowerSecondary',
        'Zasadnicze zawodowe': 'Vocational',
        'Średnie': 'Secondary',
        'Wyższe': 'Higher',
        'Brak danych/nieustalony': np.nan,
        'Nie dotyczy': np.nan
    },
    'WorkInfo': {
        'Brak danych/nieustalono': np.nan,
        'Uczeń': 'Student',
        'Student': 'Student',
        'Rolnik': 'Agriculturalist',
        'Pracujący na własny rachunek/samodzielna działalność gospodarcza': 'Employed',
        'Praca stała': 'Employed',
        'Praca dorywcza': 'Employed',
        'Bezrobotny': 'Unemployed'
    },
    'Income': {
        'Brak danych/nieustalony': np.nan,
        'Na utrzymaniu innej osoby': 'Dependent',
        'Praca': 'Steady',
        'Emerytura': 'Benefits',
        'Renta': 'Benefits',
        'Zasiłek/alimenty': 'Benefits',
        'Nie ma stałego źródła utrzymania': 'NoSteady'
    },
    'Fatal': {
        'T': 1,
        'N': 0
    },
    'Place': {
        'Droga/ulica/chodnik': 'Road',
        'Zabudowania gospodarcze': 'UtilitySpaces',
        'Mieszkanie/dom': 'House',
        'Teren kolei/tory': 'Railway',
        'Park, las': 'Forest',
        'Garaż/piwnica/strych': 'House',
        'Rzeka/jezioro/inny zbiornik wodny': 'WaterRes',
        'Zakład pracy': 'Work',
        'Placówka lecznicza lub sanatoryjna': 'Institution',
        'Miejsce prawnej izolacji': 'Isolation',
        'Obiekt wojskowy': 'PoliceArmy',
        'Placówka wychowawczo-opiekuńcza': 'Institution',
        'Szkoła/uczelnia': 'School',
        'Obiekt policyjny': 'PoliceArmy',
        'Inne': 'Other'
    },
    'Method': {
        'Rzucenie się pod pojazd w ruchu': 'Vehicle',
        'Rzucenie się z wysokości': 'Jumping',
        'Powieszenie się': 'Hanging',
        'Uszkodzenie układu krwionośnego': 'SelfHarm',
        'Zastrzelenie się/użycie broni palnej': 'Shooting',
        'Samookaleczenie powierzchowne': 'SelfHarm',
        'Zażycie środków nasennych/leków psychotropowych': 'Drugs',
        'Zatrucie gazem/spalinami': 'Gas',
        'Zażycie innych leków': 'Drugs',
        'Zatrucie środkami chemicznymi/toksycznymi': 'Poisoning',
        'Zatrucie środkami odurzającymi': 'Drugs',
        'Zatrucie dopalaczami': 'Drugs',
        'Utonięcie/utopienie się': 'Drowning',
        'Samopodpalenie': 'SelfHarm',
        'Uduszenie się': 'Other',
        'Inny': 'Other'
    },
    'Substance': {
        'Brak danych/nieustalony': np.nan,
        'Trzeźwy(a)': 'Sober',
        'Pod wpływem alkoholu': 'Alco',
        'Pod wpływem zastępczych środków/substancji (dopalaczy)': 'OtherSub',
        'Pod wpływem leków': 'OtherSub',
        'Pod wpływem środków odurzających': 'OtherSub',
        'Pod wpływem alkoholu i zastępczych środków/substancji (dopalaczy)': 'AlcoOtherSub',
        'Pod wpływem alkoholu zastępczych środków/substancji (dopalaczy)': 'AlcoOtherSub',
        'Pod wpływem alkoholu i leków': 'AlcoOtherSub',
        'Pod wpływem alkoholu i środków odurzających': 'AlcoOtherSub',
        'Pod wpływem leków i środków odurzających': 'OtherSub',
        'Pod wpływem alkoholu, leków i środków odurzających': 'AlcoOtherSub'
    },
    'Context': {
        'Nieustalony': np.nan,
        'Zawód miłosny': 'HeartBreak',
        'Leczony(a) psychiatrycznie': 'MentalHealth',
        'Nieporozumienie rodzinne/przemoc w rodzinie': 'FamilyConflict',
        'Nosiciel wirusa HIV, chory na AIDS': 'HealthLoss',
        'Nagła utrata źródła utrzymania': 'Finances',
        'Złe warunki ekonomiczne/długi': 'Finances',
        'Choroba psychiczna/zaburzenia psychiczne': 'MentalHealth',
        'Problemy w szkole lub pracy': 'SchoolWork',
        'Śmierć bliskiej osoby': 'CloseDeath',
        'Dokonanie przestępstwa lub wykroczenia': 'Crime',
        'Trwałe kalectwo': 'Disability',
        'Niepożądana ciąża': 'Other',
        'Choroba fizyczna': 'HealthLoss',
        'Pogorszenie lub nagła utrata zdrowia': 'HealthLoss',
        'Konflikt z osobami spoza rodziny': 'SchoolWork',
        'Zagrożenie lub utrata miejsca zamieszkania': 'Finances',
        'Mobbing, cybermobbing, znęcanie': 'SchoolWork',
        'Inny niewymieniony powyżej': 'Other'
    },
    'AbuseInfo1': {
        'Leczony(a) psychiatrycznie': np.nan,
        'Nadużywał(a) alkoholu': 'Alco',
        'Leczony(a) z powodu alkoholizmu': 'Alco',
        'Leczony(a) z powodu narkomanii': 'OtherSub',
        'Leczony(a) z powodu alkoholizmu i narkomanii': 'AlcoOtherSub'
    },
    'AbuseInfo2': {
        'Brak danych/nieustalono': np.nan,
        'Nadużywał(a) alkoholu': 'Alco',
        'Leczony(a) psychiatrycznie': np.nan,
        'Leczony(a) z powodu alkoholizmu': 'Alco',
        'Choroba fizyczna': np.nan,
        'Trwałe kalectwo': np.nan,
        'Zatrzymany(a) w izbie wytrzeźwień': 'Alco',
        'Nadużywał(a) alkoholu i narkotyków': 'AlcoOtherSub',
        'Nadużywał(a) alkoholu i nakrotyków': 'AlcoOtherSub',
        'Leczony(a) z powodu narkomanii': 'OtherSub',
        'Używał dopalaczy i narkotyków': 'OtherSub',
        'Nadużywał(a) alkoholu i narkotykó': 'OtherSub',
        'Nadużywał(a) alkoholu, dopalaczy i narkotyków': 'AlcoOtherSub',
        'Nadużywał(a) alkoholu, narkotyków i dopalaczy': 'AlcoOtherSub',
        'Nadużywał(a) alkoholu i dopalaczy': 'AlcoOtherSub',
        'Używał dopalaczy': 'OtherSub',
        'Nadużywał(a) alkoholu, dopalaczy, narkotyków': 'AlcoOtherSub',
        'Leczony(a) psychiatrycznie, nadużywał(a) alkoholu': 'Alco'
    }
}


In [29]:
def map_column(df: pd.DataFrame, old_col: str, new_col: str) -> pd.DataFrame:
    """Rename a column in the DataFrame.
    
    Args:
        df: Input DataFrame
        old_col: Original column name
        new_col: New column name
    
    Returns:
        DataFrame with renamed column
    """
    if old_col in df.columns:
        df.rename(columns={old_col: new_col}, inplace=True)
    return df

In [30]:
def map_columns(df: pd.DataFrame, column_mappings: dict) -> pd.DataFrame:
    """Rename multiple columns in the DataFrame based on a mapping dictionary.
    
    Args:
        df: Input DataFrame
        column_mappings: Dictionary mapping old column names to new ones
    
    Returns:
        DataFrame with renamed columns
    """
    for old_col, new_col in column_mappings.items():
        df = map_column(df, old_col, new_col)
    return df

In [31]:
def load_data(file_path: str, is_excel: bool = True):
    # Load dataset from the specified file path.
    if is_excel:
        df = pd.read_excel(file_path)
    else:
        df = pd.read_csv(file_path, low_memory=False)
    return df

In [32]:
def map_features(df: pd.DataFrame, column_mappings: dict, value_mappings: dict) -> pd.DataFrame:
    """Map multiple features using provided column and value mappings.
    
    Args:
        df: Input DataFrame
        column_mappings: Dictionary mapping old column names to new ones
        value_mappings: Dictionary of dictionaries with value mappings for each column
    
    Returns:
        DataFrame with all specified columns mapped
    """
    for old_col, new_col in column_mappings.items():
        if new_col in value_mappings:
            df[new_col] = df[new_col].map(value_mappings[new_col])
    
    # Replace values in Context columns
    context_columns = ['Context1', 'Context2', 'Context3', 'Context4']
    if 'Context' in value_mappings:
        for context_col in context_columns:
            if context_col in df.columns:
                df[context_col] = df[context_col].map(value_mappings['Context'])
    
    return df

In [33]:
excel_file_path = DATA_DIR / 'raw' / 'Samobojstwa_2023.xlsx'

In [34]:
df_raw = load_data(excel_file_path, is_excel=True)

In [35]:
df = df_raw.copy()

In [36]:
# Assuming df is your DataFrame and COLUMN_MAPPINGS is defined as shown
current_columns = df.columns.tolist()
mapped_columns = COLUMN_MAPPINGS.keys()

# Find columns in df that are not in COLUMN_MAPPINGS
unmapped_columns = [col for col in current_columns if col not in mapped_columns]

print("Columns in df that are not mentioned in COLUMN_MAPPINGS:")
print(unmapped_columns)

Columns in df that are not mentioned in COLUMN_MAPPINGS:
['Klasa miejscowości', 'W ciągu ostatniego miesiąca sprawca zdarzenia miał przynajmniej jeden raz kontakt z *']


In [37]:
df = map_columns(df, COLUMN_MAPPINGS)

In [38]:
# Assuming df is your DataFrame
column_types = {}

for column in df.columns:
    # Get unique types in the column
    unique_types = set(df[column].dropna().apply(type))
    column_types[column] = unique_types

print("Unique types of values in each column:")
for col, types in column_types.items():
    print(f"{col}: {types}")

Unique types of values in each column:
ID: {<class 'int'>, <class 'str'>}
Date: {<class 'pandas._libs.tslibs.timestamps.Timestamp'>}
AgeGroup: {<class 'str'>}
Gender: {<class 'str'>}
Marital: {<class 'str'>}
Education: {<class 'str'>}
WorkInfo: {<class 'str'>}
Income: {<class 'str'>}
Fatal: {<class 'str'>}
Place: {<class 'str'>}
Klasa miejscowości: {<class 'str'>}
Method: {<class 'str'>}
Context1: {<class 'str'>}
Context2: {<class 'str'>}
Context3: {<class 'str'>}
Context4: {<class 'str'>}
Substance: {<class 'str'>}
AbuseInfo2: {<class 'str'>}
AbuseInfo1: {<class 'str'>}
W ciągu ostatniego miesiąca sprawca zdarzenia miał przynajmniej jeden raz kontakt z *: {<class 'str'>}


In [39]:
df = map_features(df, COLUMN_MAPPINGS, VALUE_MAPPINGS)

In [40]:
# Checking for unmapped values
unmapped_values = {}

for column in df.columns:
    if column in VALUE_MAPPINGS:
        # Get the mapping for the current column
        valid_values = set(map(str, VALUE_MAPPINGS[column].values()))  # Convert valid values to strings
        # Find unique values in the column that are not in the valid values
        column_values = set(map(str, df[column].dropna().unique()))  # Convert column values to strings
        unmapped = column_values - valid_values
        
        if unmapped:
            unmapped_values[column] = unmapped

print("Unmapped values in each column:")
for col, values in unmapped_values.items():
    print(f"{col}: {values}")

Unmapped values in each column:
Fatal: {'1.0', '0.0'}


In [41]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    # Clean ID column
    if 'ID' in df.columns:
        df = df[~df['ID'].isna() & (df['ID'].str.strip() != '')]

    # Process dates if present
    if 'Date' in df.columns:
        # Convert 'YYYYMM' format to datetime
        df['Date'] = pd.to_datetime(df['Date'], format='%Y%m', errors='coerce')  # Coerce invalid formats to NaT
        # Check for any remaining NaT values and convert them from standard date format
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')  # Convert valid date strings to datetime

        # Create year and month columns
        df['DateY'] = df['Date'].dt.strftime('%Y')
        df['DateM'] = df['Date'].dt.strftime('%m')
        df['Date'] = df['DateM'] + '.' + df['DateY']

    # Find context columns dynamically
    context_columns = [col for col in df.columns if col.startswith('Context')]
    
    # Extract unique context values
    context_values = set()
    for col in context_columns:
        if col in df.columns:
            context_values.update(df[col].dropna().unique())

    # Create new binary columns for each unique context value
    for value in context_values:
        column_name = f'Context_{value}'
        df[column_name] = df[context_columns].apply(lambda row: 1 if value in row.values else 0, axis=1)

    # Drop original context columns
    df.drop(columns=context_columns, inplace=True, errors='ignore')

    # Merge AbuseInfo columns
    abuse_info_columns = [col for col in df.columns if col.startswith('AbuseInfo')]
    if abuse_info_columns:
        if len(abuse_info_columns) == 1:
            df['AbuseInfo'] = df[abuse_info_columns[0]]  # Directly assign the single column
        else:
            df['AbuseInfo'] = df[abuse_info_columns].bfill(axis=1).iloc[:, 0]  # Use backfill to fill NaNs
        df.drop(columns=abuse_info_columns, inplace=True, errors='ignore')
    
    return df

In [42]:
df = clean_data(df)

C:\Users\huber\AppData\Local\Temp\ipykernel_5540\145311925.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Date'] = pd.to_datetime(df['Date'], format='%Y%m', errors='coerce')  # Coerce invalid formats to NaT
C:\Users\huber\AppData\Local\Temp\ipykernel_5540\145311925.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Date'] = pd.to_datetime(df['Date'], errors='coerce')  # Convert valid date strings to datetime
C:\Users\huber\AppData\Local\Temp\ipykernel_5540\145311925.py:14: SettingWithCopyWarn